In [11]:
import numpy as np
import math, random
import cvxpy as cp


In [10]:
def solve ():
    eta = cp.Variable(noneg=True)
    constraints = [eta >= 0, eta <= 1]
    p12, p13, p23 = cp.Variable(noneg=True), cp.Variable(noneg=True), cp.Variable(noneg=True)
    constraints += [p12 + p13 + p23 == 1]



    for (s,t) in [(1,2), (1,3), (2,3)]:
        # solve the joint compatibility SDP for the pair (s,t)
        # M_st, J_st_a1 = 

_IncompleteInputError: incomplete input (4116847198.py, line 11)

In [ ]:
import cvxpy as cp
import numpy as np
from itertools import product


def deterministic_strategies(num_measurements=3, num_outcomes=2):
    """
    Returns all deterministic response functions λ.

    For 3 binary measurements:
        λ = (a1, a2, a3)
    where ax ∈ {0,1}.
    """
    return list(product(range(num_outcomes), repeat=num_measurements))


def D_lambda(strategy, a, x):
    """
    Deterministic distribution D_λ(a|x).

    strategy[x] is the deterministic output for measurement x.
    """
    return 1.0 if strategy[x] == a else 0.0


def noisy_binary_povm(operator, eta, d):
    """
    Binary noisy POVM:
        M_{0|x} = 1/2 (I + η A)
        M_{1|x} = 1/2 (I - η A)
    """
    I = np.eye(d)
    return [
        0.5 * (I + eta * operator),
        0.5 * (I - eta * operator),
    ]


def solve_triplet_joint_measurement(
    d=2,
    observables=None,
    solver=cp.MOSEK,
):
    """
    SDP corresponding to Eq. (11) in the screenshot.

    We maximize η such that the noisy measurements admit
    a decomposition into pairwise jointly measurable sets.

    observables:
        list of 3 Hermitian observables [A1, A2, A3]
    """

    if observables is None:
        # Pauli X,Y,Z
        sigma_x = np.array([[0, 1], [1, 0]], dtype=complex)
        sigma_y = np.array([[0, -1j], [1j, 0]], dtype=complex)
        sigma_z = np.array([[1, 0], [0, -1]], dtype=complex)

        observables = [sigma_x, sigma_y, sigma_z]

    A1, A2, A3 = observables

    I = np.eye(d)

    # ============================================================
    # Variables
    # ============================================================

    eta = cp.Variable(nonneg=True)

    # Measurement operators M_{a|x}
    # x = 0,1,2 ; a = 0,1
    M = [[None for _ in range(3)] for _ in range(2)]

    # Pairwise jointly measurable POVMs:
    # J12[a][x]
    # J23[a][x]
    # J13[a][x]
    #
    # each is a dxd PSD matrix
    J12 = [[cp.Variable((d, d), hermitian=True) for x in range(3)]
           for a in range(2)]

    J23 = [[cp.Variable((d, d), hermitian=True) for x in range(3)]
           for a in range(2)]

    J13 = [[cp.Variable((d, d), hermitian=True) for x in range(3)]
           for a in range(2)]

    # probabilities p12,p23,p13
    p12 = cp.Variable(nonneg=True)
    p23 = cp.Variable(nonneg=True)
    p13 = cp.Variable(nonneg=True)

    # deterministic decomposition weights
    strategies = deterministic_strategies(3, 2)
    n_lambda = len(strategies)

    E12 = cp.Variable(n_lambda, nonneg=True)
    E23 = cp.Variable(n_lambda, nonneg=True)
    E13 = cp.Variable(n_lambda, nonneg=True)

    # ============================================================
    # Define noisy measurements
    # ============================================================

    obs = [A1, A2, A3]

    for x in range(3):

        M[0][x] = 0.5 * (I + eta * obs[x])
        M[1][x] = 0.5 * (I - eta * obs[x])

    # ============================================================
    # Constraints
    # ============================================================

    constraints = []

    # η ≤ 1
    constraints += [eta <= 1]

    # convex mixture probabilities
    constraints += [
        p12 + p23 + p13 == 1
    ]

    # ============================================================
    # Main decomposition:
    #
    # M_{a|x} = J12 + J23 + J13
    # ============================================================

    for a in range(2):
        for x in range(3):

            constraints += [
                M[a][x]
                == J12[a][x] + J23[a][x] + J13[a][x]
            ]

    # ============================================================
    # PSD constraints
    # ============================================================

    for a in range(2):
        for x in range(3):

            constraints += [
                J12[a][x] >> 0,
                J23[a][x] >> 0,
                J13[a][x] >> 0,
            ]

    # ============================================================
    # Pairwise normalization conditions
    # ============================================================

    # ---------- J12 ----------
    for x in [0, 1]:

        constraints += [
            J12[0][x] + J12[1][x] == p12 * I
        ]

    # ---------- J23 ----------
    for x in [1, 2]:

        constraints += [
            J23[0][x] + J23[1][x] == p23 * I
        ]

    # ---------- J13 ----------
    for x in [0, 2]:

        constraints += [
            J13[0][x] + J13[1][x] == p13 * I
        ]

    # ============================================================
    # Deterministic decomposition constraints
    #
    # J^{ij}_{a|x} = Σ_λ D_λ(a|x) E^{ij}_λ
    #
    # E^{ij}_λ are scalar weights in the screenshot,
    # here promoted to PSD operators.
    # ============================================================

    for a in range(2):

        # ----- J12 -----
        for x in [0, 1]:

            rhs = 0
            for lam, strat in enumerate(strategies):
                rhs += D_lambda(strat, a, x) * E12[lam] * I

            constraints += [
                J12[a][x] == rhs
            ]

        # ----- J23 -----
        for x in [1, 2]:

            rhs = 0
            for lam, strat in enumerate(strategies):
                rhs += D_lambda(strat, a, x) * E23[lam] * I

            constraints += [
                J23[a][x] == rhs
            ]

        # ----- J13 -----
        for x in [0, 2]:

            rhs = 0
            for lam, strat in enumerate(strategies):
                rhs += D_lambda(strat, a, x) * E13[lam] * I

            constraints += [
                J13[a][x] == rhs
            ]

    # ============================================================
    # normalization of E variables
    # ============================================================

    constraints += [
        cp.sum(E12) == p12,
        cp.sum(E23) == p23,
        cp.sum(E13) == p13,
    ]

    # ============================================================
    # Solve SDP
    # ============================================================

    prob = cp.Problem(cp.Maximize(eta), constraints)

    val = prob.solve()

    print("status =", prob.status)
    print("optimal eta =", val)

    print("\np12 =", p12.value)
    print("p23 =", p23.value)
    print("p13 =", p13.value)

    print("\nE12 =", E12.value)
    print("\nE23 =", E23.value)
    print("\nE13 =", E13.value)

    return {
        "eta": eta.value,
        "p12": p12.value,
        "p23": p23.value,
        "p13": p13.value,
        "E12": E12.value,
        "E23": E23.value,
        "E13": E13.value,
        "status": prob.status,
    }


# ============================================================
# Example usage
# ============================================================

sigma_x = np.array([[0, 1], [1, 0]], dtype=complex)

sigma_y = np.array([[0, -1j], [1j, 0]], dtype=complex)

sigma_z = np.array([[1, 0], [0, -1]], dtype=complex)

result = solve_triplet_joint_measurement(
    d=2,
    observables=[sigma_x, sigma_y, sigma_z],
)

status = optimal
optimal eta = 0.3333333547209358

p12 = 0.33331286722622827
p23 = 0.3333435663952212
p13 = 0.33334356639522084

E12 = [0.04166411 0.04166411 0.04166411 0.04166411 0.04166411 0.04166411
 0.04166411 0.04166411]

E23 = [0.04166795 0.04166795 0.04166795 0.04166795 0.04166795 0.04166795
 0.04166795 0.04166795]

E13 = [0.04166795 0.04166795 0.04166795 0.04166795 0.04166795 0.04166795
 0.04166795 0.04166795]


In [ ]:
print((math.sqrt(2) + 1) / 3)

0.8047378541243649


: 

In [1]:
"""
SDP implementation of equations (10), (11), and (12) from Appendix C of

    Quintino, Budroni, Woodhead, Cabello, Cavalcanti,
    "Device-independent tests of structures of measurement incompatibility"
    arXiv:1902.05841

These SDPs decide / quantify whether a set of three POVMs is **genuinely
triplewise incompatible**, i.e. whether it CANNOT be written as a convex
combination of three sets that are pairwise compatible on the three possible
pairs (1,2), (1,3), (2,3) — Definition 1 in the paper.

Variables (shared across the three SDPs)
----------------------------------------
For each pair (s, t) of measurements in {(0,1), (0,2), (1,2)} (0-indexed),
we introduce a joint POVM
        E^{st} = { E^{st}_{a_s, a_t} : a_s, a_t = 0, ..., n_a - 1 },
all elements PSD.  By construction the marginals
        J^{st}_{a|s} = sum_{a_t} E^{st}_{a, a_t},
        J^{st}_{a|t} = sum_{a_s} E^{st}_{a_s, a}
make the measurements x = s and x = t jointly measurable inside this branch.

For the *third* measurement (the one not in {s, t}) we introduce a free
positive operator family  J^{st}_{a|x_third}  (variables only in eq. (10)/(11);
eliminated in (12) by absorption into a PSD inequality on the noisy M).

The probability p_{st} of the convex combination satisfies
        sum_a J^{st}_{a|x} = p_{st} * I   for every x.
"""

from __future__ import annotations

import cvxpy as cp
import numpy as np


# ----------------------------------------------------------------------
# 0.  Utility: build three noisy Pauli measurements (the running example)
# ----------------------------------------------------------------------

def noisy_pauli_measurements(eta: float) -> np.ndarray:
    """
    Three noisy Pauli measurements with white-noise parameter ``eta``:

        M^eta_{a|x} = eta * Pi_{a|x} + (1 - eta) * I/2

    where x = 0, 1, 2  ->  X, Y, Z  and  Pi_{a|x}  are their eigenprojectors.

    Returns an array of shape (3, 2, 2, 2): ``M[x, a]`` is the operator
    M_{a|x} on Hilbert-space dimension d = 2.
    """
    I = np.eye(2, dtype=complex)
    paulis = [
        np.array([[0, 1],  [1, 0]],  dtype=complex),   # X
        np.array([[0, -1j], [1j, 0]], dtype=complex),  # Y
        np.array([[1, 0],  [0, -1]], dtype=complex),   # Z
    ]
    M = np.zeros((3, 2, 2, 2), dtype=complex)
    for x in range(3):
        Pi0 = (I + paulis[x]) / 2
        Pi1 = (I - paulis[x]) / 2
        M[x, 0] = eta * Pi0 + (1 - eta) * I / 2
        M[x, 1] = eta * Pi1 + (1 - eta) * I / 2
    return M


# ----------------------------------------------------------------------
# Internal helper: build the (s, t) pair list and "third" index map
# ----------------------------------------------------------------------

_PAIRS = [(0, 1), (0, 2), (1, 2)]


def _third(pair: tuple[int, int]) -> int:
    """Return the index in {0, 1, 2} that is not in ``pair``."""
    return [k for k in (0, 1, 2) if k not in pair][0]


def _marginal(E_pair, pair_key, x_target, a_target, n_a):
    """Sum-marginal of ``E_pair`` onto ``x_target`` at outcome ``a_target``."""
    i, j = pair_key
    if x_target == i:
        return sum(E_pair[a_target][a_other] for a_other in range(n_a))
    if x_target == j:
        return sum(E_pair[a_other][a_target] for a_other in range(n_a))
    raise ValueError(f"x_target={x_target} is not in pair {pair_key}")


# ----------------------------------------------------------------------
# 1.  Equation (10) — feasibility SDP
# ----------------------------------------------------------------------

def feasibility_eq10(M: np.ndarray, solver: str = "CLARABEL",
                     verbose: bool = False) -> bool:
    """
    Decide whether three POVMs ``M`` are *not* genuinely triplewise
    incompatible (i.e. whether the SDP in eq. (10) is feasible).

    Parameters
    ----------
    M : ndarray, shape (3, n_a, d, d)
        ``M[x, a]`` is the operator M_{a|x}.
    solver, verbose : passed to cvxpy.

    Returns
    -------
    bool
        True  -> SDP feasible        -> NOT genuinely triplewise incompatible.
        False -> SDP infeasible      -> GENUINELY triplewise incompatible.
    """
    _, n_a, d, _ = M.shape
    I = np.eye(d)

    # Joint POVMs E^{st}: PSD d x d variables, indexed E[(s,t)][a_s][a_t].
    E = {p: [[cp.Variable((d, d), hermitian=True) for _ in range(n_a)]
             for _ in range(n_a)] for p in _PAIRS}
    # Free positive operators for the "third" measurement of each branch.
    J_third = {p: [cp.Variable((d, d), hermitian=True) for _ in range(n_a)]
               for p in _PAIRS}
    # Branch probabilities p_{st} >= 0.
    p = {pair: cp.Variable(nonneg=True) for pair in _PAIRS}

    cons = []

    # 1a. PSD on all variables.
    for pair in _PAIRS:
        for a_s in range(n_a):
            for a_t in range(n_a):
                cons.append(E[pair][a_s][a_t] >> 0)
        for a in range(n_a):
            cons.append(J_third[pair][a] >> 0)

    # 1b. Normalisation.
    for pair in _PAIRS:
        total_E = sum(E[pair][a_s][a_t]
                      for a_s in range(n_a) for a_t in range(n_a))
        cons.append(total_E == p[pair] * I)
        cons.append(sum(J_third[pair][a] for a in range(n_a)) == p[pair] * I)

    # 1c. M_{a|x} = J^{01}_{a|x} + J^{02}_{a|x} + J^{12}_{a|x}.
    for x in range(3):
        for a in range(n_a):
            terms = []
            for pair in _PAIRS:
                if x in pair:
                    terms.append(_marginal(E[pair], pair, x, a, n_a))
                else:
                    terms.append(J_third[pair][a])
            cons.append(sum(terms) == M[x, a])

    prob = cp.Problem(cp.Minimize(0), cons)
    prob.solve(solver=solver, verbose=verbose)
    return prob.status in ("optimal", "optimal_inaccurate")


# ----------------------------------------------------------------------
# 2.  Equation (11) — white-noise robustness (un-simplified)
# ----------------------------------------------------------------------

def robustness_eq11(M: np.ndarray, solver: str = "CLARABEL",
                    verbose: bool = False) -> tuple[float, str]:
    """
    Maximum eta such that the white-noise mixture

        eta * M_{a|x}  +  (1 - eta) * tr(M_{a|x}) * I/d

    is *not* genuinely triplewise incompatible.  This is equation (11) in
    the paper, with the J^3_{a|x} feasibility constraint of eq. (10) inlined.

    Returns
    -------
    eta_value : float       -- the optimal eta found
    status    : str         -- cvxpy solver status
    """
    _, n_a, d, _ = M.shape
    I = np.eye(d)

    E = {p: [[cp.Variable((d, d), hermitian=True) for _ in range(n_a)]
             for _ in range(n_a)] for p in _PAIRS}
    J_third = {p: [cp.Variable((d, d), hermitian=True) for _ in range(n_a)]
               for p in _PAIRS}
    p = {pair: cp.Variable(nonneg=True) for pair in _PAIRS}
    eta = cp.Variable()

    cons = []

    for pair in _PAIRS:
        for a_s in range(n_a):
            for a_t in range(n_a):
                cons.append(E[pair][a_s][a_t] >> 0)
        for a in range(n_a):
            cons.append(J_third[pair][a] >> 0)

    for pair in _PAIRS:
        total_E = sum(E[pair][a_s][a_t]
                      for a_s in range(n_a) for a_t in range(n_a))
        cons.append(total_E == p[pair] * I)
        cons.append(sum(J_third[pair][a] for a in range(n_a)) == p[pair] * I)

    # sum_of_J == eta * M + (1-eta) * tr(M) * I / d
    for x in range(3):
        for a in range(n_a):
            terms = []
            for pair in _PAIRS:
                if x in pair:
                    terms.append(_marginal(E[pair], pair, x, a, n_a))
                else:
                    terms.append(J_third[pair][a])
            noisy = eta * M[x, a] + (1 - eta) * np.trace(M[x, a]) * I / d
            cons.append(sum(terms) == noisy)

    prob = cp.Problem(cp.Maximize(eta), cons)
    prob.solve(solver=solver, verbose=verbose)
    return float(eta.value), prob.status


# ----------------------------------------------------------------------
# 3.  Equation (12) — simplified white-noise robustness
# ----------------------------------------------------------------------

def robustness_eq12(M: np.ndarray, solver: str = "CLARABEL",
                    verbose: bool = False) -> tuple[float, str]:
    """
    Same problem as (11) but with the ``J_third`` variables eliminated.

    The trick (described between (11) and (12) in the paper):

        noisy_M_{a|x} - J^{sx}_{a|x} - J^{tx}_{a|x}  =  J^{st}_{a|x}  >= 0,

    so we drop the equality and replace it with a PSD inequality.
    Likewise  sum_lambda E^{st}_lambda = (I/d) * sum_lambda tr E^{st}_lambda
    plays the role of  p_{st} I,  and  (1/d) sum_{pairs,lambda} tr E = 1
    enforces p_{12} + p_{13} + p_{23} = 1.
    """
    _, n_a, d, _ = M.shape
    I = np.eye(d)

    E = {pair: [[cp.Variable((d, d), hermitian=True) for _ in range(n_a)]
                for _ in range(n_a)] for pair in _PAIRS}
    eta = cp.Variable()

    cons = []

    # E^{st}_lambda >= 0.
    for pair in _PAIRS:
        for a_s in range(n_a):
            for a_t in range(n_a):
                cons.append(E[pair][a_s][a_t] >> 0)

    # sum_lambda E^{st}_lambda  ==  (I/d) * sum_lambda tr E^{st}_lambda.
    total_trace = 0
    for pair in _PAIRS:
        sum_E  = sum(E[pair][a_s][a_t]
                     for a_s in range(n_a) for a_t in range(n_a))
        sum_tr = sum(cp.real(cp.trace(E[pair][a_s][a_t]))
                     for a_s in range(n_a) for a_t in range(n_a))
        cons.append(sum_E == (I / d) * sum_tr)
        total_trace = total_trace + sum_tr

    # Branch-probabilities sum to one:  (1/d) * total trace == 1.
    cons.append(total_trace / d == 1)

    # PSD inequality: noisy_M_{a|x}  >>  J^{sx}_{a|x} + J^{tx}_{a|x}.
    # The three (s, t, x) triples cover the three pairs containing x.
    for (s, t, x) in [(0, 1, 2), (0, 2, 1), (1, 2, 0)]:
        pair_sx = tuple(sorted((s, x)))
        pair_tx = tuple(sorted((t, x)))
        for a in range(n_a):
            J_sx = _marginal(E[pair_sx], pair_sx, x, a, n_a)
            J_tx = _marginal(E[pair_tx], pair_tx, x, a, n_a)
            noisy = eta * M[x, a] + (1 - eta) * np.trace(M[x, a]) * I / d
            cons.append(noisy >> J_sx + J_tx)

    prob = cp.Problem(cp.Maximize(eta), cons)
    prob.solve(solver=solver, verbose=verbose)
    return float(eta.value), prob.status


# ----------------------------------------------------------------------
# 4.  Demo / sanity checks
# ----------------------------------------------------------------------

def _demo():
    print("=" * 70)
    print("SDP tests of genuine triplewise incompatibility")
    print("arXiv:1902.05841, Appendix C, equations (10) / (11) / (12)")
    print("=" * 70)

    # Theoretical thresholds for noisy Paulis  M^eta_{a|x} = eta * Pi + (1-eta) I/2
    eta_3wise   = 1 / np.sqrt(3)            # triplewise compatible up to here
    eta_pair    = 1 / np.sqrt(2)            # pairwise compatible up to here
    eta_genuine = (np.sqrt(2) + 1) / 3      # genuine-triplewise threshold

    print("\nNoisy-Pauli thresholds:")
    print(f"  triplewise compat.:        eta <= 1/sqrt(3)        ~= {eta_3wise:.4f}")
    print(f"  pairwise compat.:          eta <= 1/sqrt(2)        ~= {eta_pair:.4f}")
    print(f"  not genuinely 3-wise inc.: eta <= (sqrt(2)+1)/3    ~= {eta_genuine:.4f}")

    # ---- Feasibility, eq. (10) ----
    print("\n--- Feasibility check via equation (10) ---")
    for e in [0.50, 0.70, 0.80, eta_genuine, 0.81, 0.85, 1.00]:
        ok = feasibility_eq10(noisy_pauli_measurements(e))
        tag = "feasible    -> NOT genuinely 3-wise incompatible" if ok \
              else "infeasible  -> GENUINELY 3-wise incompatible"
        print(f"  eta = {e:.4f}  :  {tag}")

    # ---- Robustness, eq. (12) ----
    print("\n--- White-noise robustness via equation (12) ---")
    print("  (eta*  scales the measurement; on noisy Paulis the threshold is")
    print("   eta * eta_input <= (sqrt(2)+1)/3,  so we expect eta* = "
          "(sqrt(2)+1)/(3 * eta_input).)")
    for e in [0.5, 0.7, 0.8, eta_genuine, 0.85, 1.0]:
        val, _ = robustness_eq12(noisy_pauli_measurements(e))
        print(f"  eta_input = {e:.4f}  :  eta* = {val:.4f}   "
              f"(expected ~ {eta_genuine / e:.4f})")

    # ---- Cross-check (11) vs (12) on sharp Paulis ----
    print("\n--- Cross-check (11) vs (12) on sharp Paulis (eta_input = 1) ---")
    Msharp = noisy_pauli_measurements(1.0)
    v11, _ = robustness_eq11(Msharp)
    v12, _ = robustness_eq12(Msharp)
    print(f"  equation (11):  eta* = {v11:.6f}")
    print(f"  equation (12):  eta* = {v12:.6f}")
    print(f"  expected      :  eta* = {eta_genuine:.6f}   "
          f"= (sqrt(2)+1)/3")



In [2]:
# import cvxpy as cp
print(cp.__version__, hasattr(cp, "promote"))

1.8.2 True


In [3]:
# import cvxpy as cp, numpy as np
X = cp.Variable((2, 2), hermitian=True)
c = (X >> 0)        # if this raises AttributeError, the kernel is sick
print("OK:", c)

OK: var1 + Promote(-0.0, (2, 2)) >> 0


In [4]:
_demo()

SDP tests of genuine triplewise incompatibility
arXiv:1902.05841, Appendix C, equations (10) / (11) / (12)

Noisy-Pauli thresholds:
  triplewise compat.:        eta <= 1/sqrt(3)        ~= 0.5774
  pairwise compat.:          eta <= 1/sqrt(2)        ~= 0.7071
  not genuinely 3-wise inc.: eta <= (sqrt(2)+1)/3    ~= 0.8047

--- Feasibility check via equation (10) ---
  eta = 0.5000  :  feasible    -> NOT genuinely 3-wise incompatible
  eta = 0.7000  :  feasible    -> NOT genuinely 3-wise incompatible
  eta = 0.8000  :  feasible    -> NOT genuinely 3-wise incompatible
  eta = 0.8047  :  feasible    -> NOT genuinely 3-wise incompatible
  eta = 0.8100  :  infeasible  -> GENUINELY 3-wise incompatible
  eta = 0.8500  :  infeasible  -> GENUINELY 3-wise incompatible
  eta = 1.0000  :  infeasible  -> GENUINELY 3-wise incompatible

--- White-noise robustness via equation (12) ---
  (eta*  scales the measurement; on noisy Paulis the threshold is
   eta * eta_input <= (sqrt(2)+1)/3,  so we expect eta